# IoT telemetry processing

This notebook reads the committed telemetry JSON Lines file using the exact dataset schema,
quarantines out-of-range readings, and publishes curated hourly gold aggregates.


In [ ]:
ENVIRONMENT = "dev"
PROCESSING_DATE = "2026-04-12"
PIPELINE_RUN_ID = "local-demo"
environment = ENVIRONMENT
processing_date = PROCESSING_DATE
pipeline_run_id = PIPELINE_RUN_ID


In [ ]:
from delta.tables import DeltaTable
from pyspark.sql import functions as F, types as T, Window
import json
import uuid

batch_id = f"iot-{processing_date}-{uuid.uuid4()}"

source_path = 'Files/landing/iot-telemetry/telemetry.jsonl'
bronze_table = 'bronze.iot_telemetry_raw'
silver_table = 'silver.iot_telemetry_curated'
quarantine_table = 'silver.iot_telemetry_quarantine'
gold_table = 'gold.iot_device_hourly'


In [ ]:
def log_event(stage, message, **metrics):
    record = {
        'stage': stage,
        'message': message,
        'environment': environment,
        'processing_date': processing_date,
        'pipeline_run_id': pipeline_run_id,
        'batch_id': batch_id,
        'metrics': metrics,
    }
    print(json.dumps(record, sort_keys=True))

telemetry_schema = T.StructType([
    T.StructField('device_id', T.StringType(), False),
    T.StructField('site_id', T.StringType(), False),
    T.StructField('event_time', T.TimestampType(), False),
    T.StructField('ingest_time', T.TimestampType(), False),
    T.StructField('temperature_c', T.DoubleType(), False),
    T.StructField('vibration_mm_s', T.DoubleType(), False),
    T.StructField('humidity_pct', T.DoubleType(), False),
    T.StructField('battery_pct', T.DoubleType(), False),
    T.StructField('status', T.StringType(), False),
    T.StructField('synthetic_record', T.StringType(), False),
    T.StructField('anomaly_labels', T.ArrayType(T.StringType(), containsNull=False), False),
])

for schema_name in ['bronze', 'silver', 'gold']:
    spark.sql(f'CREATE SCHEMA IF NOT EXISTS {schema_name}')

raw_df = (
    spark.read.format('json')
    .schema(telemetry_schema)
    .load(source_path)
    .withColumn('metadata_ingestion_timestamp', F.current_timestamp())
    .withColumn('source_file', F.input_file_name())
    .withColumn('metadata_batch_id', F.lit(batch_id))
    .withColumn('metadata_pipeline_run_id', F.lit(pipeline_run_id))
    .withColumn('metadata_processing_date', F.lit(processing_date))
    .withColumn('event_date', F.to_date('event_time'))
)

raw_df.write.format('delta').mode('overwrite').partitionBy('event_date').option('overwriteSchema', 'true').saveAsTable(bronze_table)


In [ ]:
cutoff_date = F.to_date(F.lit(processing_date))
telemetry_df = spark.table(bronze_table).filter(F.col('event_date') <= cutoff_date)
event_window = Window.partitionBy('device_id', 'event_time').orderBy(F.col('ingest_time').desc_nulls_last(), F.col('metadata_ingestion_timestamp').desc_nulls_last())

deduped_df = (
    telemetry_df
    .withColumn('event_rank', F.row_number().over(event_window))
    .filter(F.col('event_rank') == 1)
    .drop('event_rank')
    .withColumn('event_hour', F.date_trunc('hour', 'event_time'))
)

quarantine_condition = (
    (F.size('anomaly_labels') > 0)
    | (F.col('temperature_c') < -40)
    | (F.col('temperature_c') > 125)
    | (F.col('vibration_mm_s') < 0)
    | (F.col('vibration_mm_s') > 50)
    | (F.col('humidity_pct') < 0)
    | (F.col('humidity_pct') > 100)
    | (F.col('battery_pct') < 0)
    | (F.col('battery_pct') > 100)
)

quarantine_df = (
    deduped_df
    .filter(quarantine_condition)
    .withColumn('quarantine_reason', F.concat_ws('; ', F.array_join('anomaly_labels', '; '), F.lit('out_of_range_reading')))
    .withColumn('quarantine_timestamp', F.current_timestamp())
)

valid_df = deduped_df.filter(~quarantine_condition)

if spark.catalog.tableExists(silver_table):
    target = DeltaTable.forName(spark, silver_table)
    (
        target.alias('target')
        .merge(valid_df.alias('source'), 'target.device_id = source.device_id AND target.event_time = source.event_time')
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )
else:
    valid_df.write.format('delta').mode('overwrite').partitionBy('event_date').option('overwriteSchema', 'true').saveAsTable(silver_table)

quarantine_df.write.format('delta').mode('overwrite').partitionBy('event_date').option('overwriteSchema', 'true').saveAsTable(quarantine_table)


In [ ]:
hourly_df = (
    valid_df
    .groupBy('event_hour', 'device_id', 'site_id', 'status')
    .agg(
        F.count('*').alias('event_count'),
        F.round(F.avg('temperature_c'), 2).alias('avg_temperature_c'),
        F.round(F.max('temperature_c'), 2).alias('max_temperature_c'),
        F.round(F.avg('vibration_mm_s'), 2).alias('avg_vibration_mm_s'),
        F.round(F.avg('humidity_pct'), 2).alias('avg_humidity_pct'),
        F.round(F.min('battery_pct'), 2).alias('min_battery_pct'),
    )
    .withColumn('processing_date', F.lit(processing_date))
    .withColumn('pipeline_run_id', F.lit(pipeline_run_id))
)

hourly_df.write.format('delta').mode('overwrite').partitionBy('processing_date').option('overwriteSchema', 'true').saveAsTable(gold_table)

log_event(
    'iot_validation',
    'IoT telemetry processing complete',
    bronze_rows=raw_df.count(),
    silver_rows=valid_df.count(),
    quarantine_rows=quarantine_df.count(),
    gold_rows=hourly_df.count(),
)
spark.sql(f'SELECT COUNT(*) AS iot_gold_rows FROM {gold_table}').show()
